In [ ]:
!pip install requests beautifulsoup4 ollama

^C


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\alice\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json
import ollama
import re
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# [Configuration] 모델 및 설정
# ==========================================
#MODEL_FAST = "llama3.1:8b"   # 영어 요약, 분류, 구조화
MODEL_SMART = "qwen2.5:7b"  # 한국어 번역, 정제, 비평
MAX_CRITIC_RETRIES = 1 

# ==========================================
# [Part 1] Scraping & Cleaning Helpers
# ==========================================

def extract_date(text):
    """텍스트에서 날짜 추출 (예: December 11, 2025)"""
    months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    date_pattern = rf'({months}\s+\d{{1,2}},\s+\d{{4}})'
    match = re.search(date_pattern, text)
    if match:
        return match.group(1)
    return None

def extract_relevant_content(text):
    """[핵심] LATEST DEVELOPMENTS 이후 내용에서 광고/불필요 섹션 정교하게 제거"""
    
    # 1. 날짜 추출
    article_date = extract_date(text)
    
    # 2. LATEST DEVELOPMENTS 이후 내용만 가져오기
    start_marker = "LATEST DEVELOPMENTS"
    start_idx = text.find(start_marker)
    if start_idx != -1:
        text = text[start_idx:]
    
    # 제외할 섹션 시작 키워드들
    exclude_markers = [
        "PRESENTED BY", "TOGETHER WITH", "AI TRAINING",
        "Trending AI Tools", "Community AI workflows",
        "Highlights: News, Guides & Events", "That's it for today!"
    ]
    
    # 포함할 섹션 (제외 마커보다 우선)
    include_markers = ["Everything else in AI today"]
    
    lines = text.split('\n')
    result_lines = []
    
    # 날짜가 있으면 맨 앞에 추가
    if article_date:
        result_lines.append(f"Date: {article_date}")
        result_lines.append("")
    
    skip_section = False
    
    for line in lines:
        stripped_line = line.strip()
        
        # 포함할 섹션 체크
        should_include = False
        for marker in include_markers:
            if marker in stripped_line:
                should_include = True
                skip_section = False
                break
        
        if should_include:
            result_lines.append(line)
            continue
        
        # 제외할 섹션 체크
        should_skip = False
        for marker in exclude_markers:
            if stripped_line.startswith(marker) or marker in stripped_line:
                should_skip = True
                skip_section = True
                break
        
        if should_skip:
            continue
            
        # 새로운 주요 섹션 시작 시 skip 해제 (대문자 헤더)
        if stripped_line.isupper() and len(stripped_line) > 3 and ' ' in stripped_line:
            is_exclude = any(marker in stripped_line for marker in exclude_markers)
            if not is_exclude:
                skip_section = False
        
        if not skip_section:
            result_lines.append(line)
    
    return '\n'.join(result_lines)

def clean_text_structure(text):
    """HTML 잔여물 및 URL 파라미터 청소"""
    text = re.sub(r'(&utm_source=[^)\s]*)', '', text)
    text = re.sub(r'(\?utm_source=[^)\s]*)', '', text)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    return text.strip()

def get_links_from_archive(page_num=1):
    url = f"https://www.therundown.ai/archive?page={page_num}"
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200: return []
        soup = BeautifulSoup(response.content, 'html.parser')
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if '/p/' in href and 'archive' not in href:
                full_link = href if href.startswith('http') else f"https://www.therundown.ai{href}"
                if full_link not in links: links.append(full_link)
        return links
    except Exception as e:
        print(f"Error fetching archive: {e}")
        return []

def scrape_article(url):
    """URL에서 기사 내용 스크래핑 및 정제"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        for script in soup(["script", "style", "nav", "footer"]): 
            script.decompose()
            
        for a in soup.find_all('a', href=True):
            if a.get_text(strip=True): 
                a.replace_with(f" {a.get_text(strip=True)} ({a['href']}) ")
                
        raw_text = soup.get_text(separator='\n', strip=True)
        
        # [수정됨] 단순 find가 아니라 extract_relevant_content 함수를 사용하여 정교하게 추출
        processed_content = extract_relevant_content(raw_text)
        
        # 후처리 (UTM 제거 등)
        final_text = clean_text_structure(processed_content)
        
        # 날짜 추출 (extract_relevant_content 내부에서 처리했지만 메타데이터용으로 한번 더 확인)
        article_date = extract_date(raw_text) or "Unknown Date"
        
        return {
            "full_text": final_text, 
            "url": url, 
            "date": article_date
        }
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

def clean_json_output(response_text):
    text = response_text.strip()
    if text.startswith('```json'): text = text[7:]
    elif text.startswith('```'): text = text[3:]
    if text.endswith('```'): text = text[:-3]
    return text.strip()

# ==========================================
# [Part 2] Agents
# ==========================================

def agent_extractor(full_text, date):
    print("  [1] Extraction Agent running...")
    prompt = f"""
    You are an expert AI News Data Extractor. 
    Your goal is to split the provided text into individual news items based on specific structural rules.

    ## Context Info
    Article Date: {date}

    ## Structural Rules (CRITICAL)
    1. **Type 1: Main Articles**
       - Look for section headers (e.g., "NOUS RESEARCH", "AI WEARABLES").
       - The Source URL usually appears in parentheses right after the title.
    
    2. **Type 2: Brief Articles**
       - Look under the section "Everything else in AI today".
       - Format: "Company/Topic [verb] (URL) description".
       - The Source URL appears immediately after the title/verb.
       - Extract EACH brief item as a separate entry.

    ## Extraction Rules
    - **source_name**: Extract the publisher/company name (e.g., "Microsoft", "DeepSeek").
    - **source_url**: Extract the specific link to the article/paper. DO NOT use the newsletter's archive link.

    ## Output Format
    Return a JSON Array ONLY. No markdown.
    [
      {{
        "raw_title": "Original Title",
        "raw_content": "Full content text of this item any excluding url",
        "source_url": "https://extracted-url.com",
        "source_name": "Publisher Name"
      }}
    ]

    ## Text to Process:
    {full_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.1, "num_predict": 4096})
    try: return json.loads(clean_json_output(response['response']))
    except: return []

def agent_summarizer_en(item_text):
    prompt = f"""
    You are an AI News Summarizer.
    
    ## Task
    Summarize the provided news text into English.

    ## Constraints
    1. Length: STRICTLY 2-3 sentences.
    2. Content: Focus on the 'who', 'what', and 'why'.
    3. Style: Professional and objective.

    Input Text:
    {item_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.2})
    return response['response'].strip()

def agent_summarizer_kr(item_text):
    prompt = f"""
    You are an expert AI Translator and Summarizer.
    
    ## Task
    Translate and summarize the provided news text into Korean.

    ## Constraints
    1. Length: 2-3 sentences.
    2. Style: Professional AI news style.
    3. Content: Capture the key technical facts accurately.

    Input Text:
    {item_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.2})
    return response['response']

def agent_classifier(item_text):
    prompt = f"""
    You are an AI Content Classifier. Classify the text using ONLY the allowed lists below.
    
    ## Allowed Categories (Select all that apply)
    - "Business", "Tech/AI", "Healthcare/Science", "Entertainment/Creative", 
    - "Education", "Society/Policy", "Hardware", "Lifestyle"
    
    ## Allowed ProductServices (Select all that apply)
    - "Text AI", "Image AI", "Video AI", "Voice AI", "Agent AI", 
    - "Automation AI", "Multimodal AI", "Vibe Coding AI", "Robotics"
    
    ## Allowed CoreElements (Select all that apply)
    - "Data", "Algorithm", "Compute", "Safety/Ethics"
    
    ## Task
    1. Analyze the text.
    2. Extract 3-5 lowercase keywords.
    3. Select relevant tags from the lists above.
    
    ## Output JSON Format
    {{
      "categories": ["Category1", ...],
      "productServices": ["Service1", ...],
      "coreElements": ["Element1", ...],
      "keywords": ["tag1", "tag2", "tag3"]
    }}
    
    Text to Classify:
    {item_text}
    """
    
    try:
        # [수정 1] format='json'을 options 밖으로 뺐습니다. (중요)
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",  
            options={"temperature": 0.1}
        )
        
        raw_text = response['response']
        
        # [디버깅] 실제 모델이 뭘 뱉었는지 눈으로 확인 (에러 시 원인 파악용)
        # print(f"DEBUG - Raw Output: {raw_text[:100]}...") 

        # [수정 2] 마크다운 코드 블록(```json ... ```)이 있으면 제거
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except json.JSONDecodeError as e:
        print(f"JSON Parsing Error: {e}")
        print(f"Model Output was: {response['response']}") # 에러나면 전체 출력 확인
        return {}
    except Exception as e:
        print(f"General Error: {e}")
        return {}

def agent_refiner_en(raw_data, summary_en, classification, feedback=""):
    """
    [수정 완료] JSON 파싱 에러 해결 버전
    """
    instruction = "Finalize the news item in English, strictly following the JSON schema."
    if feedback:
        instruction += f"\nCRITICAL FEEDBACK TO FIX: {feedback}"

    prompt = f"""
    # Role: AI Tech News Editor
    {instruction}

    # Input Data
    - Original Title: {raw_data['raw_title']}
    - Source: {raw_data['source_name']}
    - Draft Summary: {summary_en}
    - Classification: {json.dumps(classification)}

    # Rules
    1. **Title**: Rewrite the original title in a different way (creative paraphrase).
    2. **Summary**: Ensure it is 2-3 sentences.
    3. **Metrics**: Initialize to 0.

    # Final Output Schema (Strict JSON)
    {{
      "title": "Paraphrased English Title",
      "summary": "English summary...",
      "source": "{raw_data['source_name']}",
      "sourceUrl": "{raw_data['source_url']}",
      "publishedDate": "December 11, 2025", 
      "likes": 0,
      "shareCount": 0,
      "popularityScore": 0,
      "categories": {json.dumps(classification.get('categories', []))},
      "productServices": {json.dumps(classification.get('productServices', []))},
      "coreElements": {json.dumps(classification.get('coreElements', []))},
      "searchKeywords": {json.dumps(classification.get('keywords', []))}
    }}
    """
    
    try:
        # [수정 1] format='json'을 options 딕셔너리 밖으로 뺐습니다.
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",  # <-- 여기가 핵심입니다
            options={"temperature": 0.3}
        )
        
        raw_text = response['response']
        
        # [수정 2] 혹시 모를 마크다운(```json) 제거 코드 추가
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except Exception as e:
        print(f"Refiner EN Error: {e}")
        print(f"Raw Output was: {response['response']}") # 디버깅용 출력
        return {}
        
def agent_critic_en(original_text, generated_json):
    """
    [NEW] 영어 결과물 품질 검수 (제목 의역 및 요약 길이 확인)
    """
    prompt = f"""
    # Role: English Quality Assurance Judge
    Verify if the Generated JSON meets all requirements.

    # Validation Checklist
    1. **Title Paraphrasing**: Is the title creatively rephrased? (It MUST be different from the original source title, but keep the meaning).
    2. **Summary**: Is it strictly 2-3 sentences? Is the English professional and objective?
    3. **Data Integrity**: Do 'source' and 'sourceUrl' match the original text?
    4. **Structure**: Are all fields (categories, productServices, coreElements) present and valid?

    # Input Data
    Original Text: {original_text}
    Generated JSON: {json.dumps(generated_json)}

    # Output JSON Format
    {{
      "pass": true/false,
      "feedback": "If false, provide specific instructions on what to fix (e.g., 'Title is too similar to original', 'Summary is too long'). If true, empty string."
    }}
    """
    
    try:
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            options={"temperature": 0.1}, 
            format="json"
        )
        return json.loads(response['response'])
    except Exception as e:
        print(f"Critic Error: {e}")
        # 에러 발생 시 일단 통과 처리하여 파이프라인 멈춤 방지
        return {"pass": True, "feedback": ""}

def agent_refiner_kr(raw_data, summary_kr, classification, feedback=""):
    """
    [수정 완료] JSON 파싱 에러 해결 버전 (한국어)
    """
    instruction = "Write a final polished news item in Korean for a mobile app."
    if feedback: instruction += f"\nCRITICAL FEEDBACK TO FIX: {feedback}"
    
    prompt = f"""
    # Role: Professional AI Tech Editor (Korean)
    {instruction}

    # Input Data
    - Original Title: {raw_data['raw_title']}
    - Source: {raw_data['source_name']}
    - Korean Draft Summary: {summary_kr}
    
    # Classification Data (English)
    - Categories: {json.dumps(classification.get('categories', []))}
    - Keywords (EN): {json.dumps(classification.get('keywords', []))}

    # Rules
    1. **Title**: Paraphrase creatively in Korean.
    2. **Summary**: Use '해요체' (polite informal).
    3. **Keywords**: Translate the provided English keywords into natural Korean keywords (e.g., 'LLM' -> 'LLM', 'reasoning' -> '추론', 'math' -> '수학').
    
    # Final Output Schema (Strict JSON)
    {{
      "title": "Creative Korean Title",
      "summary": "Polished Korean summary...",
      "source": "{raw_data['source_name']}",
      "sourceUrl": "{raw_data['source_url']}",
      "publishedDate": "December 11, 2025", 
      "likes": 0,
      "shareCount": 0,
      "popularityScore": 0,
      "categories": {json.dumps(classification.get('categories', []))}, 
      "productServices": {json.dumps(classification.get('productServices', []))},
      "coreElements": {json.dumps(classification.get('coreElements', []))},
      "searchKeywords": ["한국어키워드1", "한국어키워드2", "EnglishKeyword"] 
    }}
    """
    
    try:
        # [수정 1] format='json'을 options 밖으로 뺐습니다.
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",   # <-- 핵심 수정 사항
            options={"temperature": 0.4}
        )
        
        raw_text = response['response']

        # [수정 2] 마크다운 제거 코드 추가
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except Exception as e:
        print(f"Refiner KR Error: {e}")
        # 디버깅을 위해 모델이 실제로 뱉은 텍스트를 출력
        print(f"Raw Output was: {response['response']}") 
        return {}

def agent_critic(original_text, generated_json):
    """
    [수정 완료] JSON 파싱 에러 해결 및 안전장치 추가 버전
    """
    prompt = f"""
    # Role: Quality Assurance Judge
    Verify if the Generated JSON meets all requirements.

    # Validation Checklist
    1. **Title**: Is it a creative Korean paraphrase? (Not just direct translation)
    2. **Summary**: Is it 2-3 sentences long? Is it in natural Korean?
    3. **Data Integrity**: Do 'source' and 'sourceUrl' match the original?
    4. **Structure**: Are all fields (categories, productServices, coreElements) present?

    Original Text: {original_text}
    Generated JSON: {json.dumps(generated_json, ensure_ascii=False)}

    # Output JSON
    {{
      "pass": true/false,
      "feedback": "If false, provide specific instructions on what to fix (e.g., 'Summary is too long', 'Title is boring'). If true, empty string."
    }}
    """
    
    try:
        # [수정 1] format='json'을 options 밖으로 뺐습니다.
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",  
            options={"temperature": 0.1}
        )
        
        raw_text = response['response']
        
        # [수정 2] 마크다운 제거 코드 추가
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except Exception as e:
        print(f"Critic Error: {e}")
        print(f"Raw Output was: {response['response']}")
        
        # [수정 3] 에러 발생 시, 무한 루프를 방지하기 위해 일단 '실패'로 처리하되, 시스템 에러임을 알림
        return {"pass": False, "feedback": "System Error: Critic output was not valid JSON. Please retry."}
# ==========================================
# [Part 3] Orchestration Engine
# ==========================================

def process_single_news_item(item, article_date):
    print(f"  Processing item: {item.get('raw_title', 'No Title')[:30]}...")
    
    # 1. Sequential Execution (순차 실행 추천)
    print("    - Summarizing (EN)...")
    summary_en = agent_summarizer_en(item['raw_content'])

    print("    - Summarizing (KR)...")
    summary_draft_kr = agent_summarizer_kr(item['raw_content'])

    print("    - Classifying...")
    classification = agent_classifier(item['raw_content'])
    
    # ==========================================
    # [Part 2] English Output Loop (NEW)
    # ==========================================
    attempts_en = 0
    feedback_en = ""
    eng_final = None
    
    print("    - Refining English...")
    while attempts_en <= MAX_CRITIC_RETRIES:
        # feedback_en을 넘겨주도록 호출
        refined_json_en = agent_refiner_en(item, summary_en, classification, feedback_en)
        
        # 영어 Critic 호출
        judge_result_en = agent_critic_en(item['raw_content'], refined_json_en)
        
        if judge_result_en['pass']:
            eng_final = refined_json_en
            print(f"    ✅ English Passed (Attempt {attempts_en+1})")
            break
        else:
            print(f"    ❌ English Failed: {judge_result_en['feedback']}")
            feedback_en = judge_result_en['feedback']
            attempts_en += 1
    
    if eng_final is None:
        print("    ⚠️ Forced accept English")
        eng_final = refined_json_en # 실패해도 마지막 결과 사용
        
    eng_output = eng_final
    eng_output['publishedDate'] = article_date 

    # ==========================================
    # [Part 3] Korean Output Loop (Existing)
    # ==========================================
    attempts_kr = 0
    feedback_kr = ""
    kor_final = None
    
    print("    - Refining Korean...")
    while attempts_kr <= MAX_CRITIC_RETRIES:
        refined_json_kr = agent_refiner_kr(item, summary_draft_kr, classification, feedback_kr)
        judge_result_kr = agent_critic(item['raw_content'], refined_json_kr)
        
        if judge_result_kr['pass']:
            kor_final = refined_json_kr
            print(f"    ✅ Korean Passed (Attempt {attempts_kr+1})")
            break
        else:
            print(f"    ❌ Korean Failed: {judge_result_kr['feedback']}")
            feedback_kr = judge_result_kr['feedback']
            attempts_kr += 1
            
    if kor_final is None:
        print("    ⚠️ Forced accept Korean")
        kor_final = refined_json_kr
        
    kor_output = kor_final
    kor_output['publishedDate'] = article_date 
    
    return eng_output, kor_output

In [ ]:
item = {'raw_title': "Nous Research's AI takes on elite math exam",
  'raw_content': 'Nous Research just open-sourced Nomos 1, a new 30B parameter reasoning system that scored 87 out of 120 on the 2025 Putnam Contest — crushing rivals like Qwen 3 on one of the most prestigious collegiate math competitions. The system uses a two-phase approach: AI ‘workers’ solve and self-critique responses, with a tournament-style bracket then selecting the best submission. Nomos’ score would have placed second among nearly 4,000 human competitors last year, with the model earning eight perfect problem scores. Nous also released and open-sourced a reasoning harness — orchestration code that manages how the model solves problems. Running Qwen3 through the same harness and setup scored just 24/120, with the result showing gains coming from model training rather than the harness.',
  'source_url': 'https://x.com/NousResearch/status/1998536543565127968?s=20',
  'source_name': 'Nous Research'}
article_date = 'December 11, 2025'
eng_output, kor_output = process_single_news_item(item, article_date)
eng_output

  Processing item: Nous Research's AI takes on el...
    - Summarizing (EN)...
    - Summarizing (KR)...
    - Classifying...
    - Refining English...
    ❌ English Failed: Title is too similar to original. The summary is also too long and lacks the self-critique aspect of Nomos' approach. Ensure that all fields are present and valid.
    ❌ English Failed: ['Title is too similar to original. The title should be creatively paraphrased.', 'Summary is too long. It exceeds the 3-sentence limit and contains unnecessary details.', "All fields are present, but 'categories', 'productServices', and 'coreElements' might not fully align with the content."]
    ❌ English Failed: ['Title is not creatively paraphrased. The title should be different from the original source title while keeping a similar meaning.', 'Summary is too short and does not fully capture the details of the original text, such as the two-phase approach used by Nomos and its comparison with Qwen3.']
    ⚠️ Forced accept Englis

{'title': 'AI System Nomos Outperforms Humans in Elite Math Competition',
 'summary': "Nous Research's open-sourced AI, Nomos, excelled in the 2025 Putnam Contest, outscoring nearly 4,000 human competitors.",
 'source': 'Nous Research',
 'sourceUrl': 'https://x.com/NousResearch/status/1998536543565127968?s=20',
 'publishedDate': 'December 11, 2025',
 'likes': 0,
 'shareCount': 0,
 'popularityScore': 0,
 'categories': ['Tech/AI'],
 'productServices': ['Text AI', 'Multimodal AI'],
 'coreElements': ['Algorithm', 'Compute', 'Safety/Ethics'],
 'searchKeywords': ['nomos', 'ai reasoning system', 'putnam contest']}

In [ ]:
kor_output

{'title': '수학 문제 해결, AI가 전문가와 경쟁하다',
 'summary': "Nous Research은 최근 Nomos 1이라는 새로운 30B 파라미터 추론 시스템을 오픈소스로 공개했으며, 이 시스템은 2025년 Putnam 대회에서 뛰어난 성적을 거두었습니다. Nomos는 AI '워커'들이 문제를 해결한 후 자가 검토해 최고 제출물을 선정하는 방식으로 작동합니다.",
 'source': 'Nous Research',
 'sourceUrl': 'https://x.com/NousResearch/status/1998536543565127968?s=20',
 'publishedDate': 'December 11, 2025',
 'likes': 0,
 'shareCount': 0,
 'popularityScore': 0,
 'categories': ['Tech/AI'],
 'productServices': ['Text AI', 'Multimodal AI'],
 'coreElements': ['Algorithm', 'Compute', 'Safety/Ethics'],
 'searchKeywords': ['AI 수학 문제 해결', 'Nomos AI 시스템', 'Putnam 대회']}

In [ ]:
def main_pipeline(start_page=1, end_page=22):
    """
    여러 페이지를 처리하고 각 페이지별로 별도 JSON 파일 저장
    """
    print("=== AI News Pipeline Started (Production Mode) ===")
    
    # 각 페이지별로 처리
    for page_num in range(start_page, end_page + 1):
        print(f"\n{'='*60}")
        print(f"Processing Page {page_num}")
        print('='*60)
        
        # 1. 링크 수집
        links = get_links_from_archive(page_num=page_num)
        
        if not links:
            print(f"No links found on page {page_num}. Skipping.")
            continue

        # 결과를 모을 리스트 (페이지마다 초기화)
        all_english_results = []
        all_korean_results = []

        print(f"Found {len(links)} newsletters on page {page_num}. Starting batch process...")

        for i, target_url in enumerate(links):
            print(f"\n[{i+1}/{len(links)}] Processing Newsletter: {target_url}")
            
            # 2. 스크래핑
            article_data = scrape_article(target_url)
            if not article_data: 
                print("  -> Scraping failed. Skipping.")
                continue

            # 3. 추출 (Extractor)
            raw_items = agent_extractor(article_data['full_text'], article_data['date'])
            print(f"  -> Extracted {len(raw_items)} items.")
            
            # 4. 뉴스 아이템별 처리
            for idx, item in enumerate(raw_items):
                try:
                    eng_result, kor_result = process_single_news_item(item, article_data['date'])
                    if eng_result:
                        all_english_results.append(eng_result)
                    if kor_result:
                        all_korean_results.append(kor_result)
                except Exception as e:
                    print(f"    Error processing item {idx}: {e}")
                    continue

        # 페이지별 파일 저장
        filename_en = f"data/data_en/news_english_p{page_num}.json"
        filename_kr = f"data/data_ko/news_korean_p{page_num}.json"
        
        with open(filename_en, 'w', encoding='utf-8') as f:
            json.dump(all_english_results, f, ensure_ascii=False, indent=2)
            
        with open(filename_kr, 'w', encoding='utf-8') as f:
            json.dump(all_korean_results, f, ensure_ascii=False, indent=2)
            
        print(f"\n✅ Page {page_num} Completed: {len(all_english_results)} items")
        print(f"   Saved to: {filename_en} & {filename_kr}")
        
    print(f"\n=== All Pages Processed ===")


if __name__ == "__main__":
    # 페이지 1~2 처리 (각각 별도 파일로 저장)
    main_pipeline(start_page=1, end_page=22)

=== AI News Pipeline Started (Production Mode) ===

Processing Page 1
Found 12 newsletters on page 1. Starting batch process...

[1/12] Processing Newsletter: https://www.therundown.ai/p/disneys-billion-dollar-ai-bet-on-openai
  [1] Extraction Agent running...
  -> Extracted 7 items.
  Processing item: OpenAI, Disney lock in $1B lic...
    - Summarizing (EN)...
    - Summarizing (KR)...
    - Classifying...
    - Refining English...
    ❌ English Failed: ['Title is too similar to original. The title should be creatively paraphrased.', 'Summary is too long. It exceeds the 2-3 sentence limit.', "Structure: 'publishedDate' field is not required and does not match the input data format."]
    ❌ English Failed: ['Title is too similar to original. The title should be creatively rephrased while keeping the meaning.', 'Summary is too long. It exceeds the 3-sentence limit and could be more concise.', "Structure: 'publishedDate' field is missing."]
    ⚠️ Forced accept English
    - Refining Kor

## Robot

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import json
import ollama
import re
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# [Configuration] 모델 및 설정
# ==========================================
#MODEL_FAST = "llama3.1:8b"   # 영어 요약, 분류, 구조화
MODEL_SMART = "qwen2.5:7b"  # 한국어 번역, 정제, 비평
MAX_CRITIC_RETRIES = 1 

# ==========================================
# [Part 1] Scraping & Cleaning Helpers
# ==========================================

def extract_date(text):
    """텍스트에서 날짜 추출 (예: December 11, 2025)"""
    months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    date_pattern = rf'({months}\s+\d{{1,2}},\s+\d{{4}})'
    match = re.search(date_pattern, text)
    if match:
        return match.group(1)
    return None

def extract_relevant_content(text):
    """[핵심] LATEST DEVELOPMENTS 이후 내용에서 광고/불필요 섹션 정교하게 제거"""
    
    # 1. 날짜 추출
    article_date = extract_date(text)
    
    # 2. LATEST DEVELOPMENTS 이후 내용만 가져오기
    start_marker = "LATEST DEVELOPMENTS"
    start_idx = text.find(start_marker)
    if start_idx != -1:
        text = text[start_idx:]
    
    # 제외할 섹션 시작 키워드들
    exclude_markers = [
        "PRESENTED BY", "TOGETHER WITH", "AI TRAINING",
        "Trending AI Tools", "Community AI workflows",
        "Highlights: News, Guides & Events", "That's it for today!"
    ]
    
    # 포함할 섹션 (제외 마커보다 우선)
    include_markers = ["Everything else in AI today"]
    
    lines = text.split('\n')
    result_lines = []
    
    # 날짜가 있으면 맨 앞에 추가
    if article_date:
        result_lines.append(f"Date: {article_date}")
        result_lines.append("")
    
    skip_section = False
    
    for line in lines:
        stripped_line = line.strip()
        
        # 포함할 섹션 체크
        should_include = False
        for marker in include_markers:
            if marker in stripped_line:
                should_include = True
                skip_section = False
                break
        
        if should_include:
            result_lines.append(line)
            continue
        
        # 제외할 섹션 체크
        should_skip = False
        for marker in exclude_markers:
            if stripped_line.startswith(marker) or marker in stripped_line:
                should_skip = True
                skip_section = True
                break
        
        if should_skip:
            continue
            
        # 새로운 주요 섹션 시작 시 skip 해제 (대문자 헤더)
        if stripped_line.isupper() and len(stripped_line) > 3 and ' ' in stripped_line:
            is_exclude = any(marker in stripped_line for marker in exclude_markers)
            if not is_exclude:
                skip_section = False
        
        if not skip_section:
            result_lines.append(line)
    
    return '\n'.join(result_lines)

def clean_text_structure(text):
    """HTML 잔여물 및 URL 파라미터 청소"""
    text = re.sub(r'(&utm_source=[^)\s]*)', '', text)
    text = re.sub(r'(\?utm_source=[^)\s]*)', '', text)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    return text.strip()

def get_links_from_archive(page_num=1):
    url = f"https://robotnews.therundown.ai/archive?page={page_num}"
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200: return []
        soup = BeautifulSoup(response.content, 'html.parser')
        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if '/p/' in href and 'archive' not in href:
                full_link = href if href.startswith('http') else f"https://robotnews.therundown.ai{href}"
                if full_link not in links: links.append(full_link)
        return links
    except Exception as e:
        print(f"Error fetching archive: {e}")
        return []

def scrape_article(url):
    """URL에서 기사 내용 스크래핑 및 정제"""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        for script in soup(["script", "style", "nav", "footer"]): 
            script.decompose()
            
        for a in soup.find_all('a', href=True):
            if a.get_text(strip=True): 
                a.replace_with(f" {a.get_text(strip=True)} ({a['href']}) ")
                
        raw_text = soup.get_text(separator='\n', strip=True)
        
        # [수정됨] 단순 find가 아니라 extract_relevant_content 함수를 사용하여 정교하게 추출
        processed_content = extract_relevant_content(raw_text)
        
        # 후처리 (UTM 제거 등)
        final_text = clean_text_structure(processed_content)
        
        # 날짜 추출 (extract_relevant_content 내부에서 처리했지만 메타데이터용으로 한번 더 확인)
        article_date = extract_date(raw_text) or "Unknown Date"
        
        return {
            "full_text": final_text, 
            "url": url, 
            "date": article_date
        }
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

def clean_json_output(response_text):
    text = response_text.strip()
    if text.startswith('```json'): text = text[7:]
    elif text.startswith('```'): text = text[3:]
    if text.endswith('```'): text = text[:-3]
    return text.strip()

# ==========================================
# [Part 2] Agents
# ==========================================

def agent_extractor(full_text, date):
    print("  [1] Extraction Agent running...")
    prompt = f"""
    You are an expert AI News Data Extractor. 
    Your goal is to split the provided text into individual news items based on specific structural rules.

    ## Context Info
    Article Date: {date}

    ## Structural Rules (CRITICAL)
    1. **Type 1: Main Articles**
       - Look for section headers (e.g., "NOUS RESEARCH", "AI WEARABLES").
       - The Source URL usually appears in parentheses right after the title.
    
    2. **Type 2: Brief Articles**
       - Look under the section "Everything else in AI today".
       - Format: "Company/Topic [verb] (URL) description".
       - The Source URL appears immediately after the title/verb.
       - Extract EACH brief item as a separate entry.

    ## Extraction Rules
    - **source_name**: Extract the publisher/company name (e.g., "Microsoft", "DeepSeek").
    - **source_url**: Extract the specific link to the article/paper. DO NOT use the newsletter's archive link.

    ## Output Format
    Return a JSON Array ONLY. No markdown.
    [
      {{
        "raw_title": "Original Title",
        "raw_content": "Full content text of this item any excluding url",
        "source_url": "https://extracted-url.com",
        "source_name": "Publisher Name"
      }}
    ]

    ## Text to Process:
    {full_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.1, "num_predict": 4096})
    try: return json.loads(clean_json_output(response['response']))
    except: return []


#common process
def agent_summarizer_en(item_text):
    prompt = f"""
    You are an AI News Summarizer.
    
    ## Task
    Summarize the provided news text into English.

    ## Constraints
    1. Length: STRICTLY 2-3 sentences.
    2. Content: Focus on the 'who', 'what', and 'why'.
    3. Style: Professional and objective.

    Input Text:
    {item_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.2})
    return response['response'].strip()

def agent_summarizer_kr(item_text):
    prompt = f"""
    You are an expert AI Translator and Summarizer.
    
    ## Task
    Translate and summarize the provided news text into Korean.

    ## Constraints
    1. Length: 2-3 sentences.
    2. Style: Professional AI news style.
    3. Content: Capture the key technical facts accurately.

    Input Text:
    {item_text}
    """
    response = ollama.generate(model=MODEL_SMART, prompt=prompt, options={"temperature": 0.2})
    return response['response']

def agent_classifier(item_text):
    prompt = f"""
    You are an AI Content Classifier. Classify the text using ONLY the allowed lists below.
    
    ## Allowed Categories (Select all that apply)
    - "Business", "Tech/AI", "Healthcare/Science", "Entertainment/Creative", 
    - "Education", "Society/Policy", "Hardware", "Lifestyle"
    
    ## Allowed ProductServices (Select all that apply)
    - "Text AI", "Image AI", "Video AI", "Voice AI", "Agent AI", 
    - "Automation AI", "Multimodal AI", "Vibe Coding AI", "Robotics"
    
    ## Allowed CoreElements (Select all that apply)
    - "Data", "Algorithm", "Compute", "Safety/Ethics"
    
    ## Task
    1. Analyze the text.
    2. Extract 3-5 lowercase keywords.
    3. Select relevant tags from the lists above.
    
    ## Output JSON Format
    {{
      "categories": ["Category1", ...],
      "productServices": ["Service1", ...],
      "coreElements": ["Element1", ...],
      "keywords": ["tag1", "tag2", "tag3"]
    }}
    
    Text to Classify:
    {item_text}
    """
    
    try:
        # [수정 1] format='json'을 options 밖으로 뺐습니다. (중요)
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",  
            options={"temperature": 0.1}
        )
        
        raw_text = response['response']
        
        # [디버깅] 실제 모델이 뭘 뱉었는지 눈으로 확인 (에러 시 원인 파악용)
        # print(f"DEBUG - Raw Output: {raw_text[:100]}...") 

        # [수정 2] 마크다운 코드 블록(```json ... ```)이 있으면 제거
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except json.JSONDecodeError as e:
        print(f"JSON Parsing Error: {e}")
        print(f"Model Output was: {response['response']}") # 에러나면 전체 출력 확인
        return {}
    except Exception as e:
        print(f"General Error: {e}")
        return {}

def agent_refiner_en(raw_data, summary_en, classification, feedback=""):
    """
    [수정 완료] JSON 파싱 에러 해결 버전
    """
    instruction = "Finalize the news item in English, strictly following the JSON schema."
    if feedback:
        instruction += f"\nCRITICAL FEEDBACK TO FIX: {feedback}"

    prompt = f"""
    # Role: AI Tech News Editor
    {instruction}

    # Input Data
    - Original Title: {raw_data['raw_title']}
    - Source: {raw_data['source_name']}
    - Draft Summary: {summary_en}
    - Classification: {json.dumps(classification)}

    # Rules
    1. **Title**: Rewrite the original title in a different way (creative paraphrase).
    2. **Summary**: Ensure it is 2-3 sentences.
    3. **Metrics**: Initialize to 0.

    # Final Output Schema (Strict JSON)
    {{
      "title": "Paraphrased English Title",
      "summary": "English summary...",
      "source": "{raw_data['source_name']}",
      "sourceUrl": "{raw_data['source_url']}",
      "publishedDate": "December 11, 2025", 
      "likes": 0,
      "shareCount": 0,
      "popularityScore": 0,
      "categories": {json.dumps(classification.get('categories', []))},
      "productServices": {json.dumps(classification.get('productServices', []))},
      "coreElements": {json.dumps(classification.get('coreElements', []))},
      "searchKeywords": {json.dumps(classification.get('keywords', []))}
    }}
    """
    
    try:
        # [수정 1] format='json'을 options 딕셔너리 밖으로 뺐습니다.
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",  # <-- 여기가 핵심입니다
            options={"temperature": 0.3}
        )
        
        raw_text = response['response']
        
        # [수정 2] 혹시 모를 마크다운(```json) 제거 코드 추가
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except Exception as e:
        print(f"Refiner EN Error: {e}")
        print(f"Raw Output was: {response['response']}") # 디버깅용 출력
        return {}
        
def agent_critic_en(original_text, generated_json):
    """
    [NEW] 영어 결과물 품질 검수 (제목 의역 및 요약 길이 확인)
    """
    prompt = f"""
    # Role: English Quality Assurance Judge
    Verify if the Generated JSON meets all requirements.

    # Validation Checklist
    1. **Title Paraphrasing**: Is the title creatively rephrased? (It MUST be different from the original source title, but keep the meaning).
    2. **Summary**: Is it strictly 2-3 sentences? Is the English professional and objective?
    3. **Data Integrity**: Do 'source' and 'sourceUrl' match the original text?
    4. **Structure**: Are all fields (categories, productServices, coreElements) present and valid?

    # Input Data
    Original Text: {original_text}
    Generated JSON: {json.dumps(generated_json)}

    # Output JSON Format
    {{
      "pass": true/false,
      "feedback": "If false, provide specific instructions on what to fix (e.g., 'Title is too similar to original', 'Summary is too long'). If true, empty string."
    }}
    """
    
    try:
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            options={"temperature": 0.1}, 
            format="json"
        )
        return json.loads(response['response'])
    except Exception as e:
        print(f"Critic Error: {e}")
        # 에러 발생 시 일단 통과 처리하여 파이프라인 멈춤 방지
        return {"pass": True, "feedback": ""}

def agent_refiner_kr(raw_data, summary_kr, classification, feedback=""):
    """
    [수정 완료] JSON 파싱 에러 해결 버전 (한국어)
    """
    instruction = "Write a final polished news item in Korean for a mobile app."
    if feedback: instruction += f"\nCRITICAL FEEDBACK TO FIX: {feedback}"
    
    prompt = f"""
    # Role: Professional AI Tech Editor (Korean)
    {instruction}

    # Input Data
    - Original Title: {raw_data['raw_title']}
    - Source: {raw_data['source_name']}
    - Korean Draft Summary: {summary_kr}
    
    # Classification Data (English)
    - Categories: {json.dumps(classification.get('categories', []))}
    - Keywords (EN): {json.dumps(classification.get('keywords', []))}

    # Rules
    1. **Title**: Paraphrase creatively in Korean.
    2. **Summary**: Use '해요체' (polite informal).
    3. **Keywords**: Translate the provided English keywords into natural Korean keywords (e.g., 'LLM' -> 'LLM', 'reasoning' -> '추론', 'math' -> '수학').
    
    # Final Output Schema (Strict JSON)
    {{
      "title": "Creative Korean Title",
      "summary": "Polished Korean summary...",
      "source": "{raw_data['source_name']}",
      "sourceUrl": "{raw_data['source_url']}",
      "publishedDate": "December 11, 2025", 
      "likes": 0,
      "shareCount": 0,
      "popularityScore": 0,
      "categories": {json.dumps(classification.get('categories', []))}, 
      "productServices": {json.dumps(classification.get('productServices', []))},
      "coreElements": {json.dumps(classification.get('coreElements', []))},
      "searchKeywords": ["한국어키워드1", "한국어키워드2", "EnglishKeyword"] 
    }}
    """
    
    try:
        # [수정 1] format='json'을 options 밖으로 뺐습니다.
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",   # <-- 핵심 수정 사항
            options={"temperature": 0.4}
        )
        
        raw_text = response['response']

        # [수정 2] 마크다운 제거 코드 추가
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except Exception as e:
        print(f"Refiner KR Error: {e}")
        # 디버깅을 위해 모델이 실제로 뱉은 텍스트를 출력
        print(f"Raw Output was: {response['response']}") 
        return {}

def agent_critic(original_text, generated_json):
    """
    [수정 완료] JSON 파싱 에러 해결 및 안전장치 추가 버전
    """
    prompt = f"""
    # Role: Quality Assurance Judge
    Verify if the Generated JSON meets all requirements.

    # Validation Checklist
    1. **Title**: Is it a creative Korean paraphrase? (Not just direct translation)
    2. **Summary**: Is it 2-3 sentences long? Is it in natural Korean?
    3. **Data Integrity**: Do 'source' and 'sourceUrl' match the original?
    4. **Structure**: Are all fields (categories, productServices, coreElements) present?

    Original Text: {original_text}
    Generated JSON: {json.dumps(generated_json, ensure_ascii=False)}

    # Output JSON
    {{
      "pass": true/false,
      "feedback": "If false, provide specific instructions on what to fix (e.g., 'Summary is too long', 'Title is boring'). If true, empty string."
    }}
    """
    
    try:
        # [수정 1] format='json'을 options 밖으로 뺐습니다.
        response = ollama.generate(
            model=MODEL_SMART, 
            prompt=prompt, 
            format="json",  
            options={"temperature": 0.1}
        )
        
        raw_text = response['response']
        
        # [수정 2] 마크다운 제거 코드 추가
        if "```" in raw_text:
            raw_text = raw_text.replace("```json", "").replace("```", "").strip()

        return json.loads(raw_text)

    except Exception as e:
        print(f"Critic Error: {e}")
        print(f"Raw Output was: {response['response']}")
        
        # [수정 3] 에러 발생 시, 무한 루프를 방지하기 위해 일단 '실패'로 처리하되, 시스템 에러임을 알림
        return {"pass": False, "feedback": "System Error: Critic output was not valid JSON. Please retry."}
# ==========================================
# [Part 3] Orchestration Engine
# ==========================================

def process_single_news_item(item, article_date):
    print(f"  Processing item: {item.get('raw_title', 'No Title')[:30]}...")
    
    # 1. Sequential Execution (순차 실행 추천)
    print("    - Summarizing (EN)...")
    summary_en = agent_summarizer_en(item['raw_content'])

    print("    - Summarizing (KR)...")
    summary_draft_kr = agent_summarizer_kr(item['raw_content'])

    print("    - Classifying...")
    classification = agent_classifier(item['raw_content'])
    
    # ==========================================
    # [Part 2] English Output Loop (NEW)
    # ==========================================
    attempts_en = 0
    feedback_en = ""
    eng_final = None
    
    print("    - Refining English...")
    while attempts_en <= MAX_CRITIC_RETRIES:
        # feedback_en을 넘겨주도록 호출
        refined_json_en = agent_refiner_en(item, summary_en, classification, feedback_en)
        
        # 영어 Critic 호출
        judge_result_en = agent_critic_en(item['raw_content'], refined_json_en)
        
        if judge_result_en['pass']:
            eng_final = refined_json_en
            print(f"    ✅ English Passed (Attempt {attempts_en+1})")
            break
        else:
            print(f"    ❌ English Failed: {judge_result_en['feedback']}")
            feedback_en = judge_result_en['feedback']
            attempts_en += 1
    
    if eng_final is None:
        print("    ⚠️ Forced accept English")
        eng_final = refined_json_en # 실패해도 마지막 결과 사용
        
    eng_output = eng_final
    eng_output['publishedDate'] = article_date 

    # ==========================================
    # [Part 3] Korean Output Loop (Existing)
    # ==========================================
    attempts_kr = 0
    feedback_kr = ""
    kor_final = None
    
    print("    - Refining Korean...")
    while attempts_kr <= MAX_CRITIC_RETRIES:
        refined_json_kr = agent_refiner_kr(item, summary_draft_kr, classification, feedback_kr)
        judge_result_kr = agent_critic(item['raw_content'], refined_json_kr)
        
        if judge_result_kr['pass']:
            kor_final = refined_json_kr
            print(f"    ✅ Korean Passed (Attempt {attempts_kr+1})")
            break
        else:
            print(f"    ❌ Korean Failed: {judge_result_kr['feedback']}")
            feedback_kr = judge_result_kr['feedback']
            attempts_kr += 1
            
    if kor_final is None:
        print("    ⚠️ Forced accept Korean")
        kor_final = refined_json_kr
        
    kor_output = kor_final
    kor_output['publishedDate'] = article_date 
    
    return eng_output, kor_output

In [12]:
def main_pipeline(start_page=1, end_page=7):
    """
    여러 페이지를 처리하고 각 페이지별로 별도 JSON 파일 저장
    """
    print("=== AI News Pipeline Started (Production Mode) ===")
    
    # 각 페이지별로 처리
    for page_num in range(start_page, end_page + 1):
        print(f"\n{'='*60}")
        print(f"Processing Page {page_num}")
        print('='*60)
        
        # 1. 링크 수집
        links = get_links_from_archive(page_num=page_num)
        
        if not links:
            print(f"No links found on page {page_num}. Skipping.")
            continue

        # 결과를 모을 리스트 (페이지마다 초기화)
        all_english_results = []
        all_korean_results = []

        print(f"Found {len(links)} newsletters on page {page_num}. Starting batch process...")

        for i, target_url in enumerate(links):
            print(f"\n[{i+1}/{len(links)}] Processing Newsletter: {target_url}")
            
            # 2. 스크래핑
            article_data = scrape_article(target_url)
            if not article_data: 
                print("  -> Scraping failed. Skipping.")
                continue

            # 3. 추출 (Extractor)
            raw_items = agent_extractor(article_data['full_text'], article_data['date'])
            print(f"  -> Extracted {len(raw_items)} items.")
            
            # 4. 뉴스 아이템별 처리
            for idx, item in enumerate(raw_items):
                try:
                    eng_result, kor_result = process_single_news_item(item, article_data['date'])
                    if eng_result:
                        all_english_results.append(eng_result)
                    if kor_result:
                        all_korean_results.append(kor_result)
                except Exception as e:
                    print(f"    Error processing item {idx}: {e}")
                    continue

        # 페이지별 파일 저장
        filename_en = f"data/data_en/robot_runtime_news_english_p{page_num}.json"
        filename_kr = f"data/data_ko/robot_runtime_news_korean_p{page_num}.json"
        
        with open(filename_en, 'w', encoding='utf-8') as f:
            json.dump(all_english_results, f, ensure_ascii=False, indent=2)
            
        with open(filename_kr, 'w', encoding='utf-8') as f:
            json.dump(all_korean_results, f, ensure_ascii=False, indent=2)
            
        print(f"\n✅ Page {page_num} Completed: {len(all_english_results)} items")
        print(f"   Saved to: {filename_en} & {filename_kr}")
        
    print(f"\n=== All Pages Processed ===")


if __name__ == "__main__":
    # 페이지 1~2 처리 (각각 별도 파일로 저장)
    main_pipeline(start_page=1, end_page=1)

=== AI News Pipeline Started (Production Mode) ===

Processing Page 1
Found 12 newsletters on page 1. Starting batch process...

[1/12] Processing Newsletter: https://robotnews.therundown.ai/p/skild-eyes-14b-for-robot-brains
  [1] Extraction Agent running...
  -> Extracted 12 items.
  Processing item: Skild’s robot brain draws mega...
    - Summarizing (EN)...
    - Summarizing (KR)...
    - Classifying...
    - Refining English...
    ❌ English Failed: ['Title is not different from the original source title.', "Summary is too long and includes information not present in the original text (e.g., 'TechCrunch' as a source).", 'Source and sourceUrl do not match the original text.']
    ❌ English Failed: The following issues need to be addressed:
1. The title should be different from the original source title.
2. The summary exceeds the recommended length of 2-3 sentences.
3. 'source' and 'sourceUrl' do not match the original text.
    ⚠️ Forced accept English
    - Refining Korean...
    

## Weekly AI News Data Automation